## Open In Colab

[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/risk-uq-suite/notebooks/miscalibration_interpretation_colab.ipynb)


# Risk/UQ Paper Track: Miscalibration Interpretation

## Objective
Load already-saved `miscalibration_probe` artifacts from persistent storage and convert them into a formal verdict on your hypothesis.

This notebook answers, from evidence:
1. Are planner-side risk proxies miscalibrated?
2. Do budgeted decisions become over-confident (`false-safe`) or under-confident (`safe-rejected`)?
3. Does calibration reduce those decision errors?


## Hypotheses Being Adjudicated

- `H1`: Raw planner proxy risk is miscalibrated (nominal ECE is non-trivial and/or NLL is high).
- `H2`: Under a safety budget `tau`, raw probabilities lead to unsafe acceptances (false-safe > 0).
- `H3`: Calibrated probabilities improve reliability metrics and decision-level error rates relative to raw.
- `H4`: Miscalibration/decision error worsens under non-nominal shift suites.

Output includes a claim-by-claim verdict table with `supported`, `not_supported`, or `inconclusive`.


## Step 1 - Deterministic Bootstrap Constants


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
EXPERIMENT_SLUG = 'risk-uq-suite'
EXPERIMENT_CONFIG_PATH = f'{REPO_DIR}/configs/experiments/{EXPERIMENT_SLUG}.json'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))


## Step 2 - Storage Setup


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('[storage] google.colab drive mount unavailable:', exc)

drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    raise RuntimeError('Drive root not found at /content/drive/MyDrive. Ensure Google Drive is mounted.')

required_root = Path(REQUIRED_DRIVE_FOLDER)
required_root.mkdir(parents=True, exist_ok=True)
probe = required_root / '.risk_uq_storage_probe'
probe.write_text('ok')
probe.unlink(missing_ok=True)
print('[storage] ready:', required_root)


## Step 3 - Repo Sync + Runtime Bootstrap


In [ ]:
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for p in (REPO_DIR, f'{REPO_DIR}/src'):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; restart runtime before continuing.')


## Step 4 - Configuration + Run Context


In [ ]:
from src.workflows import (
    initialize_risk_uq_run_context,
    load_experiment_config,
    report_risk_uq_run_context,
)

EXPERIMENT_CFG = load_experiment_config(
    slug=EXPERIMENT_SLUG,
    repo_root=REPO_DIR,
    default_on_missing=False,
)
run_cfg = dict(EXPERIMENT_CFG.get('run', {}))

RUN_NAME = str(run_cfg.get('run_name', '')).strip()
RUN_PREFIX = str(run_cfg.get('run_prefix', 'risk_uq')).strip() or 'risk_uq'
PERSIST_ROOT = str(run_cfg.get('persist_root', '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1')).strip()
N_SHARDS = int(max(1, int(run_cfg.get('n_shards', 1))))
SHARD_ID = run_cfg.get('shard_id', 'auto')
RESUME_FROM_EXISTING = bool(run_cfg.get('resume_from_existing', True))
RUN_ENABLED = bool(run_cfg.get('run_enabled', True))

RUN_TAG_PREFIX = str(run_cfg.get('run_tag_prefix', RUN_PREFIX)).strip() or RUN_PREFIX
RUN_MODE_CFG = str(run_cfg.get('run_mode', 'auto')).strip().lower() or 'auto'
RUN_MODE = RUN_MODE_CFG if RESUME_FROM_EXISTING else 'fresh'
RUN_TAG = RUN_NAME

run_ctx = initialize_risk_uq_run_context(
    run_tag=RUN_TAG,
    run_tag_prefix=RUN_TAG_PREFIX,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    resume_mode=RUN_MODE,
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    planner_backend='latentdriver',
)

cfg = run_ctx.cfg
search_cfg = {}
RUN_TAG = str(run_ctx.run_tag)
RUN_NAME = str(RUN_NAME or RUN_TAG)
SHARD_ID = int(run_ctx.shard_id)

# Interpretation knobs
ANALYSIS_RUN_PREFIX_OVERRIDE = ''  # optional absolute run_prefix from a previous probe run
FOCUS_LABEL = 'failure_proxy_h15'
RISK_BUDGET_TAU = 0.20
MAX_DISCOVERED_RUNS = 50

print('EXPERIMENT_CONFIG_PATH =', EXPERIMENT_CONFIG_PATH)
print('candidate run_prefix from config =', cfg.run_prefix)
print('RUN_PREFIX/RUN_NAME (contract dir) =', RUN_PREFIX, RUN_NAME)
print('RUN_ENABLED =', RUN_ENABLED, ' RESUME_FROM_EXISTING =', RESUME_FROM_EXISTING)
report_risk_uq_run_context(run_ctx, display_fn=display)


## Step 5 - Locate Available Probe Artifacts


In [ ]:
from src.workflows import (
    discover_probe_run_prefixes,
    has_existing_miscalibration_probe_artifacts,
)

analysis_run_prefix = str(ANALYSIS_RUN_PREFIX_OVERRIDE).strip() or str(cfg.run_prefix)
discovered_runs_df = discover_probe_run_prefixes(PERSIST_ROOT, limit=MAX_DISCOVERED_RUNS)

if not discovered_runs_df.empty:
    print('[artifacts] discovered probe runs (most recent first):')
    display(discovered_runs_df.head(20))
else:
    print('[artifacts] no probe runs discovered under PERSIST_ROOT:', PERSIST_ROOT)

if (not has_existing_miscalibration_probe_artifacts(analysis_run_prefix)) and (not discovered_runs_df.empty):
    analysis_run_prefix = str(discovered_runs_df.iloc[0]['run_prefix'])

print('analysis_run_prefix =', analysis_run_prefix)
print('artifacts_present =', has_existing_miscalibration_probe_artifacts(analysis_run_prefix))

if bool(RUN_ENABLED) and (not has_existing_miscalibration_probe_artifacts(analysis_run_prefix)):
    raise FileNotFoundError(
        'No completed miscalibration probe artifacts were found. '        'Run miscalibration_probe_colab.ipynb first, then re-run this notebook.'
    )


## Step 6 - Load Artifacts and Adjudicate Hypotheses


In [ ]:
from src.workflows import load_and_analyze_miscalibration_probe

interpret_bundle = None
if not bool(RUN_ENABLED):
    print('[main] skipped: RUN_ENABLED=False')
else:
    interpret_bundle = load_and_analyze_miscalibration_probe(
        run_prefix=analysis_run_prefix,
        focus_label=FOCUS_LABEL,
        threshold=RISK_BUDGET_TAU,
    )
    print(interpret_bundle.narrative)
    display(interpret_bundle.verdict_df)

    key_cols = [
        'scope', 'variant_group', 'ece', 'nll', 'brier',
        'empirical_failure_given_accepted', 'false_safe_rate',
        'safe_rejected_rate', 'budget_violated_rate',
    ]
    key_cols = [c for c in key_cols if c in interpret_bundle.metric_summary_df.columns]
    display(interpret_bundle.metric_summary_df.loc[:, key_cols])

    final_claim = interpret_bundle.verdict_df[
        interpret_bundle.verdict_df['claim'].astype(str).eq('Problem framing validated (miscalibration + decision impact)')
    ]
    if not final_claim.empty:
        print('[hypothesis verdict]', str(final_claim.iloc[0]['status']).upper())
    else:
        print('[hypothesis verdict] INCONCLUSIVE (missing final claim row)')


## Step 7 - Plots (Reliability, Shift Error, Budget Decision Diagnostics)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

figure_paths = {}

if interpret_bundle is None:
    print('[plots] skipped: no interpretation bundle')
else:
    # Plot A: reliability diagram for one nominal and one shifted suite.
    rel = interpret_bundle.reliability_focus_df.copy()
    if not rel.empty and {'mean_prob', 'event_rate', 'shift_suite', 'variant', 'variant_group'}.issubset(rel.columns):
        def _pick_variant(rel_df: pd.DataFrame, group: str, preferred: str) -> str | None:
            sub = rel_df[rel_df['variant_group'].astype(str).eq(group)]
            if sub.empty:
                return None
            if sub['variant'].astype(str).eq(preferred).any():
                return preferred
            return str(sub['variant'].astype(str).iloc[0])

        raw_variant = _pick_variant(rel, 'raw', 'planner_combo_raw')
        cal_variant = _pick_variant(rel, 'cal', 'planner_combo_platt')

        shifts = ['nominal_clean']
        non_nom = sorted([s for s in rel['shift_suite'].astype(str).unique().tolist() if s != 'nominal_clean'])
        if len(non_nom) > 0:
            shifts.append(non_nom[0])

        fig, axes = plt.subplots(1, len(shifts), figsize=(6 * len(shifts), 5), sharex=True, sharey=True)
        if len(shifts) == 1:
            axes = [axes]

        for ax, shift_suite in zip(axes, shifts):
            ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='ideal')
            for group, variant, color in [
                ('raw', raw_variant, 'tab:red'),
                ('cal', cal_variant, 'tab:blue'),
            ]:
                if variant is None:
                    continue
                sub = rel[
                    rel['shift_suite'].astype(str).eq(shift_suite)
                    & rel['variant'].astype(str).eq(variant)
                ].sort_values('mean_prob')
                if sub.empty:
                    continue
                ax.plot(
                    sub['mean_prob'].to_numpy(dtype=float),
                    sub['event_rate'].to_numpy(dtype=float),
                    marker='o',
                    linewidth=1.5,
                    label=f'{group}:{variant}',
                    color=color,
                )
            ax.set_title(f'Reliability ({shift_suite})')
            ax.set_xlabel('Mean predicted probability')
            ax.set_ylabel('Empirical failure rate')
            ax.grid(alpha=0.25)
            ax.legend(loc='best')

        fig.suptitle(f'Focus label: {FOCUS_LABEL}', y=1.02)
        fig.tight_layout()
        rel_plot_path = f'{analysis_run_prefix}_miscalibration_interpretation_reliability.png'
        fig.savefig(rel_plot_path, dpi=160, bbox_inches='tight')
        figure_paths['reliability_plot'] = rel_plot_path
        plt.show()
    else:
        print('[plots] reliability plot skipped: required columns missing')

    # Plot B: ECE/NLL/Brier by shift for raw vs cal groups.
    per = interpret_bundle.per_shift_focus_df.copy()
    if not per.empty and {'shift_suite', 'variant_group', 'ece', 'nll', 'brier'}.issubset(per.columns):
        agg = (
            per.groupby(['shift_suite', 'variant_group'], as_index=False)
            .agg(ece=('ece', 'mean'), nll=('nll', 'mean'), brier=('brier', 'mean'))
        )
        metric_names = ['ece', 'nll', 'brier']
        fig, axes = plt.subplots(1, len(metric_names), figsize=(6 * len(metric_names), 5))
        for ax, metric in zip(axes, metric_names):
            pivot = agg.pivot(index='shift_suite', columns='variant_group', values=metric)
            preferred_cols = [c for c in ['raw', 'cal', 'other'] if c in pivot.columns]
            pivot = pivot.loc[:, preferred_cols]
            pivot.plot(kind='bar', ax=ax)
            ax.set_title(metric.upper())
            ax.set_xlabel('shift_suite')
            ax.set_ylabel(metric)
            ax.grid(axis='y', alpha=0.25)
        fig.tight_layout()
        shift_plot_path = f'{analysis_run_prefix}_miscalibration_interpretation_shift_metrics.png'
        fig.savefig(shift_plot_path, dpi=160, bbox_inches='tight')
        figure_paths['shift_metrics_plot'] = shift_plot_path
        plt.show()
    else:
        print('[plots] shift metric plot skipped: required columns missing')

    # Plot C: decision-level diagnostics under budget tau.
    thr = interpret_bundle.threshold_focus_df.copy()
    required_thr = {'shift_suite', 'variant_group', 'false_safe_rate', 'safe_rejected_rate'}
    if not thr.empty and required_thr.issubset(thr.columns):
        agg_thr = (
            thr.groupby(['shift_suite', 'variant_group'], as_index=False)
            .agg(
                false_safe_rate=('false_safe_rate', 'mean'),
                safe_rejected_rate=('safe_rejected_rate', 'mean'),
                empirical_failure_given_accepted=('empirical_failure_given_accepted', 'mean'),
                accepted_rate=('accepted_rate', 'mean'),
            )
        )

        metrics = ['false_safe_rate', 'safe_rejected_rate', 'empirical_failure_given_accepted']
        fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 5))
        for ax, metric in zip(axes, metrics):
            pivot = agg_thr.pivot(index='shift_suite', columns='variant_group', values=metric)
            preferred_cols = [c for c in ['raw', 'cal', 'other'] if c in pivot.columns]
            pivot = pivot.loc[:, preferred_cols]
            pivot.plot(kind='bar', ax=ax)
            ax.set_title(metric)
            ax.set_xlabel('shift_suite')
            ax.set_ylabel(metric)
            if metric == 'empirical_failure_given_accepted':
                ax.axhline(float(RISK_BUDGET_TAU), color='k', linestyle='--', linewidth=1, label='tau')
                ax.legend(loc='best')
            ax.grid(axis='y', alpha=0.25)

        fig.tight_layout()
        thr_plot_path = f'{analysis_run_prefix}_miscalibration_interpretation_threshold_diagnostics.png'
        fig.savefig(thr_plot_path, dpi=160, bbox_inches='tight')
        figure_paths['threshold_plot'] = thr_plot_path
        plt.show()
    else:
        print('[plots] threshold diagnostic plot skipped: required columns missing')


## Step 8 - Export Verdict Artifacts + Contract Manifest


In [ ]:
from src.workflows import write_contract_storage_mirror, write_notebook_contract_manifest

if interpret_bundle is None:
    stage_name = 'miscalibration_interpretation_skipped'
    artifact_paths = {}
    metrics_path = None
    extra = {'run_skipped': 1}
else:
    verdict_path = f'{analysis_run_prefix}_miscalibration_interpretation_verdict.csv'
    metric_summary_path = f'{analysis_run_prefix}_miscalibration_interpretation_metric_summary.csv'
    per_shift_path = f'{analysis_run_prefix}_miscalibration_interpretation_per_shift_focus.csv'
    threshold_path = f'{analysis_run_prefix}_miscalibration_interpretation_threshold_focus.csv'

    interpret_bundle.verdict_df.to_csv(verdict_path, index=False)
    interpret_bundle.metric_summary_df.to_csv(metric_summary_path, index=False)
    interpret_bundle.per_shift_focus_df.to_csv(per_shift_path, index=False)
    interpret_bundle.threshold_focus_df.to_csv(threshold_path, index=False)

    artifact_paths = dict(interpret_bundle.artifact_paths)
    artifact_paths.update(
        {
            'miscalibration_interpretation_verdict': verdict_path,
            'miscalibration_interpretation_metric_summary': metric_summary_path,
            'miscalibration_interpretation_per_shift_focus': per_shift_path,
            'miscalibration_interpretation_threshold_focus': threshold_path,
            **figure_paths,
        }
    )

    stage_name = 'miscalibration_interpretation_completed'
    metrics_path = metric_summary_path
    extra = {
        'analysis_run_prefix': str(analysis_run_prefix),
        'focus_label': str(FOCUS_LABEL),
        'risk_budget_tau': float(RISK_BUDGET_TAU),
        'verdict_rows': int(len(interpret_bundle.verdict_df)),
        'metric_summary_rows': int(len(interpret_bundle.metric_summary_df)),
        'figure_count': int(len(figure_paths)),
    }

manifest_path = write_notebook_contract_manifest(
    run_prefix=cfg.run_prefix,
    run_tag=RUN_TAG,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    notebook_name='miscalibration_interpretation_colab',
    stage=stage_name,
    repo_dir=REPO_DIR,
    git_commit=globals().get('_RISK_UQ_REPO_REV', ''),
    extra_fields=extra,
)

contract_paths = write_contract_storage_mirror(
    persist_root=PERSIST_ROOT,
    run_prefix=RUN_PREFIX,
    run_name=RUN_NAME,
    run_prefix_path=cfg.run_prefix,
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    stage=stage_name,
    git_commit=str(globals().get('_RISK_UQ_REPO_REV', '')),
    resume_from_existing=bool(RESUME_FROM_EXISTING),
    run_enabled=bool(RUN_ENABLED),
    artifact_paths=artifact_paths,
    metrics_csv_path=metrics_path,
    extra_fields=extra,
)

print('manifest_path =', manifest_path)
print('contract_run_manifest =', contract_paths['contract_run_manifest'])
if artifact_paths:
    print('exported_interpretation_artifacts:')
    for k, v in sorted(artifact_paths.items()):
        print('-', k, '->', v)


## Reading The Verdict

Use the `miscalibration_interpretation_verdict.csv` file as the top-line evidence table.

- `supported`: current run artifacts back the claim.
- `not_supported`: current artifacts argue against the claim.
- `inconclusive`: missing signal or insufficient data in that run.

If final claim is inconclusive, first verify probe dataset size, split coverage, and shift-suite coverage in the upstream probe run.
